# Streamlit Multipage Demo

This notebook follows the same workflow as `05_streamlit_setting.ipynb`:

- Use `%%writefile` inside the notebook to generate Streamlit files
- Use `app.py` as the main entry point
- Put extra pages inside the `pages/` folder so Streamlit builds the sidebar navigation automatically

This demo stays small on purpose and focuses on three ideas:

1. The basic `app.py` + `pages/` multipage structure
2. Shared state across pages with `st.session_state`
3. Common UI elements such as text input, slider, chart, table, toggle, and button


In [ ]:
!pip install streamlit pandas numpy pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.4 MB/s eta 0:00:00


## Ngrok Setup Reminder

After installing `pyngrok`, remember to set up your ngrok token the same way as in the previous lesson.

1. Open https://dashboard.ngrok.com/authtokens
2. Copy your ngrok authtoken
3. In Colab, open the **Secrets** panel on the left
4. Create a secret with the key name `ngrok`
5. Paste the token as the value

The ngrok launch cell in this notebook will read that secret automatically.


## Logic

The multipage logic in Streamlit is simple:

- `streamlit run app.py` starts the main app
- Every `.py` file inside the `pages/` directory becomes a separate page in the sidebar
- `st.session_state` lets different pages share the same data, such as a username, UI preference, or feature toggle

This notebook writes three files:

- `app.py`: the home page that introduces the demo and initializes shared state
- `pages/1_Data_Explorer.py`: a small data page with controls, a chart, and a table
- `pages/2_Settings.py`: a settings page that updates and resets shared state


In [2]:
!mkdir -p pages


In [32]:
%%writefile app.py
import streamlit as st


# Configure the page before rendering any UI elements.
st.set_page_config(page_title="Streamlit Multipage Demo", page_icon="🧭", layout="wide")

# Initialize shared values once so every page can safely read them later.
if "username" not in st.session_state:
    st.session_state.username = "Streamlit Learner"
if "show_notes" not in st.session_state:
    st.session_state.show_notes = True
if "theme" not in st.session_state:
    st.session_state.theme = "Blue"

if st.session_state.theme == "Green":
    st.markdown("""
        <style>
            .stApp { background-color: #e8f5e9; }
            h1, h2, h3 { color: #2e7d32; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Orange":
    st.markdown("""
        <style>
            .stApp { background-color: #fff3e0; }
            h1, h2, h3 { color: #e65100; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Blue":
    st.markdown("""
        <style>
            .stApp { background-color: #e3f2fd; }
            h1, h2, h3 { color: #1565c0; }
        </style>
    """, unsafe_allow_html=True)

st.title("🧭 Streamlit Multipage Demo")
st.write(
    "This is the home page. `app.py` is the entry file, and every script inside `pages/` becomes a page in the sidebar automatically."
)

# The sidebar is a good place to show state that should remain visible across pages.
with st.sidebar:
    st.header("Shared State")
    st.write(f"User: {st.session_state.username}")
    st.write(f"Theme: {st.session_state.theme}")
    st.write(f"Show Notes: {st.session_state.show_notes}")

# Quick summary metrics make the demo structure easier to understand at a glance.
col1, col2, col3 = st.columns(3)
col1.metric("Pages", 3)
col2.metric("Shared Keys", len(st.session_state))
col3.metric("Framework", "Streamlit")

st.subheader("1. Update shared state on the home page")

# This input updates a value that will also be visible on the other pages.
username = st.text_input("Your name", value=st.session_state.username)
st.session_state.username = username.strip() or "Streamlit Learner"

# A toggle is a convenient way to turn explanations on or off across the whole app.
show_notes = st.toggle("Show learning notes", value=st.session_state.show_notes)
st.session_state.show_notes = show_notes

st.subheader("2. What this demo shows")
st.markdown(
    """
- Sidebar navigation is created automatically from the `pages/` folder.
- `st.session_state` lets different pages read and update the same values.
- Each page can focus on one feature area instead of one large script.
    """
)

if st.session_state.show_notes:
    st.info(
        "Suggested flow: change your name here, open `Data Explorer`, then open `Settings` to confirm that the same session state is shared across pages."
    )


Overwriting app.py


In [33]:
%%writefile pages/1_Data_Explorer.py
import numpy as np
import pandas as pd
import streamlit as st

# Each page can define its own title and layout while still participating in the same app.
st.set_page_config(page_title="Data Explorer", page_icon="📊", layout="wide")

# Recreate missing keys defensively so this page also works if opened first.
if "username" not in st.session_state:
    st.session_state.username = "Streamlit Learner"
if "show_notes" not in st.session_state:
    st.session_state.show_notes = True
if "theme" not in st.session_state:
    st.session_state.theme = "Blue"

if st.session_state.theme == "Green":
    st.markdown("""
        <style>
            .stApp { background-color: #e8f5e9; }
            h1, h2, h3 { color: #2e7d32; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Orange":
    st.markdown("""
        <style>
            .stApp { background-color: #fff3e0; }
            h1, h2, h3 { color: #e65100; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Blue":
    st.markdown("""
        <style>
            .stApp { background-color: #e3f2fd; }
            h1, h2, h3 { color: #1565c0; }
        </style>
    """, unsafe_allow_html=True)

st.title("📊 Data Explorer")
st.write(f"Hello, {st.session_state.username}. This page shows a compact example of controls, generated data, and a chart.")

# Sliders let the user change how much data is generated and keep the demo interactive.
rows = st.slider("Number of rows", min_value=10, max_value=120, value=40, step=10)
seed = st.slider("Random seed", min_value=0, max_value=20, value=7)

# A deterministic random generator makes the output reproducible for the same seed.
rng = np.random.default_rng(seed)
df = pd.DataFrame(
    {
        "day": np.arange(1, rows + 1),
        "sales": rng.integers(80, 180, size=rows),
        "cost": rng.integers(40, 120, size=rows),
    }
)

# Derive a new metric so the page demonstrates a small transformation step.
df["profit"] = df["sales"] - df["cost"]

# The selected metric controls which column is drawn in the chart.
metric = st.selectbox("Chart metric", ["sales", "cost", "profit"])
st.line_chart(df.set_index("day")[[metric]])

# Expanders keep optional detail available without crowding the layout.
with st.expander("Preview table"):
    st.dataframe(df, use_container_width=True)

# A lightweight summary block is useful when users only need high-level numbers.
summary = {
    "Total sales": int(df["sales"].sum()),
    "Total cost": int(df["cost"].sum()),
    "Total profit": int(df["profit"].sum()),
}
st.json(summary)

if st.session_state.show_notes:
    st.caption(
        "This page demonstrates how sliders affect generated data, how a select box controls the chart, and how session state keeps the username and note preference in sync across pages."
    )


Overwriting pages/1_Data_Explorer.py


In [34]:
%%writefile pages/2_Settings.py
import streamlit as st

# A separate settings page is a clean way to manage app-wide preferences.
st.set_page_config(page_title="Settings", page_icon="⚙️", layout="centered")

# Recreate missing defaults so this page is safe to open directly from the sidebar.
if "username" not in st.session_state:
    st.session_state.username = "Streamlit Learner"
if "show_notes" not in st.session_state:
    st.session_state.show_notes = True
if "theme" not in st.session_state:
    st.session_state.theme = "Blue"

if st.session_state.theme == "Green":
    st.markdown("""
        <style>
            .stApp { background-color: #e8f5e9; }
            h1, h2, h3 { color: #2e7d32; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Orange":
    st.markdown("""
        <style>
            .stApp { background-color: #fff3e0; }
            h1, h2, h3 { color: #e65100; }
        </style>
    """, unsafe_allow_html=True)
elif st.session_state.theme == "Blue":
    st.markdown("""
        <style>
            .stApp { background-color: #e3f2fd; }
            h1, h2, h3 { color: #1565c0; }
        </style>
    """, unsafe_allow_html=True)

st.title("⚙️ Settings")
st.write("This page focuses on editing and resetting shared state.")

# The select box writes directly back to session state so the value persists across pages.
st.session_state.theme = st.selectbox(
    "Theme preference",
    ["Blue", "Green", "Orange"],
    index=["Blue", "Green", "Orange"].index(st.session_state.theme),
)

# This checkbox controls whether helper notes are shown throughout the app.
st.session_state.show_notes = st.checkbox(
    "Show notes across pages",
    value=st.session_state.show_notes,
)

st.subheader("Current session state")
st.json(
    {
        "username": st.session_state.username,
        "theme": st.session_state.theme,
        "show_notes": st.session_state.show_notes,
    }
)

# A reset button is useful in demos because it returns the app to a known state.
if st.button("Reset demo state"):
    for key, value in {
        "username": "Streamlit Learner",
        "theme": "Blue",
        "show_notes": True,
    }.items():
        if key == "theme":
          st.session_state[key] =  st.session_state.theme
        else:
          st.session_state[key] = value

    st.success("State reset complete. Switch pages to see the updated values.")

st.markdown(
    """
**How it works**

- `st.session_state` behaves like a dictionary that belongs to the current user session.
- When one page updates a value, the other pages read the latest version of that same value.
- This pattern is useful for filters, lightweight preferences, login-related flags, or draft form data.
    """
)


Overwriting pages/2_Settings.py


## Run

This notebook supports two launch styles:

1. Local run: `streamlit run app.py`
2. Colab or remote run: start Streamlit and expose it with `pyngrok`

If you are running in Colab, the ngrok token should be stored in the Secrets panel with the key name `ngrok`.
If you are running outside Colab, you can also set the environment variable `NGROK_AUTHTOKEN`.


In [6]:
# Local run reminder
print("Run locally with: streamlit run app.py")


Run locally with: streamlit run app.py


In [35]:
# Optional: start Streamlit with ngrok, similar to the previous lesson.
import os
import subprocess
import time
from pyngrok import conf, ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

APP_FILE = "app.py"
PORT = 8501

# In Colab, read the secret named 'ngrok'. Outside Colab, fall back to an environment variable.
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("ngrok")
except Exception:
    NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN", "")

if not NGROK_TOKEN:
    raise ValueError("No ngrok token found. Use the Colab Secret named 'ngrok' or set NGROK_AUTHTOKEN.")

# Stop any previous tunnels so the new public URL points to the current app instance only.
ngrok.kill()

# Launch Streamlit in the background. A small delay gives the server time to bind to port 8501.
streamlit_process = subprocess.Popen(
    ["streamlit", "run", APP_FILE, "--server.port", str(PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(3)

# Create a public HTTPS URL that forwards traffic to the local Streamlit port.
# If ERR_NGROK_334 appears, the same ngrok domain is still online in another runtime or session.
try:
    public_url = ngrok.connect(
        PORT,
        pyngrok_config=conf.PyngrokConfig(auth_token=NGROK_TOKEN),
        bind_tls=True,
    ).public_url
except PyngrokNgrokHTTPError as exc:
    if "ERR_NGROK_334" in str(exc):
        raise RuntimeError(
            "ERR_NGROK_334: this ngrok endpoint is already online in another session. "
            "Stop the older Colab runtime or any other ngrok process using the same token, then run this cell again."
        ) from exc
    raise

print("=" * 60)
print(f"Streamlit URL: {public_url}")
print("=" * 60)
print("Open the URL above. If you rerun this cell, use the newest URL only.")


Streamlit URL: https://oralia-nonflammable-sonorously.ngrok-free.dev
Open the URL above. If you rerun this cell, use the newest URL only.


In [14]:
# Optional cleanup
import os
from pyngrok import ngrok

# Stop the Streamlit process and close any active ngrok tunnels.
os.system("pkill -f 'streamlit run app.py'")
ngrok.kill()
print("Stopped Streamlit and ngrok.")


Stopped Streamlit and ngrok.
